In [3]:
import yaml
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

with open(os.path.join(PROJECT_ROOT, "config", "base2.yaml"), "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

emb_name = cfg["embedding"]["use"] 
emb_conf = cfg["embedding"]["siliconflow"]
print("Using embedding:", emb_name)
print("Embedding config:", emb_conf)

Using embedding: siliconflow
Embedding config: {'api_key': 'sk-varotcommbpttwkmvrcznqgroqewaglrtcnlskyvaabgolim', 'api_base': 'https://api.siliconflow.cn/v1', 'model': 'Pro/BAAI/bge-m3'}


In [4]:
import requests

def embed_texts_siliconflow(texts, emb_conf):
    url = emb_conf["api_base"] + "/embeddings" 
    api_key = emb_conf["api_key"]
    model = emb_conf["model"]

    payload = {
        "model": model,
        "input": texts
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    r = requests.post(url, json=payload, headers=headers)
    r.raise_for_status()
    data = r.json()

    # 返回一批 embedding
    return [item["embedding"] for item in data["data"]]


In [5]:
vec = embed_texts_siliconflow(["hello"], emb_conf)[0]
print("dim =", len(vec))   # 1024

dim = 1024


In [8]:
import json

with open(os.path.join(PROJECT_ROOT, "data", "faiss", "refined_document_chunks.json"), "r", encoding="utf-8") as f:
    chunks = json.load(f)

cnt = 0
texts = [c["content"] for c in chunks]

for item in texts:
    if len(item) > 1000:
        cnt += 1 

metadatas = [
    {
        "chunk_id": c["chunk_id"],
        "title": c["title"],
        "metadata": c["metadata"],
        "path_titles": c["path_titles"],
        "path_numbering": c["path_numbering"],
    }
    for c in chunks
]

print("Chunks loaded:", len(texts))
print("long chunk:", cnt)


Chunks loaded: 11587
long chunk: 100


In [21]:
import re

def split_content(content, max_length=300):
    """
    将 content 按句子或段落切分，每小段不超过 max_length。
    """
    # 先按句子切分（中文、英文、列表项）
    sentences = re.split(r'(?<=[。！!？?])\s*', content)

    small_chunks = []
    current = ""

    for sent in sentences:
        if not sent.strip():
            continue

        # 如果加上当前句子超过 max_length，则存储当前块
        if len(current) + len(sent) > max_length:
            if current.strip():
                small_chunks.append(current.strip())
            current = sent
        else:
            current += sent

    if current.strip():
        small_chunks.append(current.strip())

    return small_chunks

In [22]:
def refine_chunks_and_metadata(chunks, max_length=300):
    refined_chunks = []
    refined_meta = []
    new_id = 1

    for c in chunks:
        content = c["content"]

        if len(content) <= max_length:
            # 不切分
            new_chunk = c.copy()
            new_chunk["chunk_id"] = new_id

            refined_chunks.append(new_chunk)
            refined_meta.append({
                "chunk_id": new_id,
                "title": new_chunk["title"],
                "content": new_chunk["content"],
                "metadata": new_chunk["metadata"],
                "path_titles": new_chunk["path_titles"],
                "path_numbering": new_chunk["path_numbering"],
            })

            new_id += 1

        else:
            # 内容太长，切成若干 part
            parts = split_content(content, max_length=max_length)

            for idx, sub in enumerate(parts):
                new_chunk = {
                    "chunk_id": new_id,
                    "title": f"{c['title']} (part {idx+1})",
                    "is_numbered": c["is_numbered"],
                    "logical_level": c["logical_level"],
                    "content": sub,
                    "metadata": c["metadata"],
                    "path_titles": c["path_titles"],
                    "path_numbering": c["path_numbering"],
                }

                refined_chunks.append(new_chunk)
                refined_meta.append({
                    "chunk_id": new_id,
                    "title": new_chunk["title"],
                    "content": sub,
                    "metadata": new_chunk["metadata"],
                    "path_titles": new_chunk["path_titles"],
                    "path_numbering": new_chunk["path_numbering"],
                })

                new_id += 1

    return refined_chunks, refined_meta

In [ ]:
refined_chunks, refined_meta = refine_chunks_and_metadata(chunks, max_length=300)

print("Refined chunks:", len(refined_chunks))
print("Refined metadata entries:", len(refined_meta))

import json

save_path = os.path.join(PROJECT_ROOT, "data", "faiss", "refined_document_chunks.json")

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(refined_meta, f, ensure_ascii=False, indent=2)

print("Metadata saved to:", save_path)

Refined chunks: 2190
Refined metadata entries: 2190
Metadata saved to: /home/guoziyang/AIgorithm_Agent/src/refined_ducument_chunks.json


In [ ]:
import json

meta_path = os.path.join(PROJECT_ROOT, "data", "faiss", "refined_document_chunks.json")

with open(meta_path, "r", encoding="utf-8") as f:
    refined_document_chunks = json.load(f)

print("Loaded metadata entries:", len(refined_document_chunks))


Loaded metadata entries: 2190


In [32]:
import numpy as np

texts = [m["content"] for m in refined_ducument_chunks]
ids = [m["chunk_id"] for m in refined_ducument_chunks]

embeddings = []
batch_size = 16

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    vecs = embed_texts_siliconflow(batch, emb_conf)
    embeddings.extend(vecs)

# 转 numpy，FAISS 需要 float32
embeddings = np.array(embeddings).astype("float32")
ids = np.array(ids).astype("int64")

print("Embedding shape:", embeddings.shape)
print("Example dimension:", len(embeddings[0]))


Embedding shape: (2190, 1024)
Example dimension: 1024


In [33]:
import faiss

dim = embeddings.shape[1]  # 1024
index = faiss.IndexFlatL2(dim)
index = faiss.IndexIDMap(index)
index.add_with_ids(embeddings, ids)

print("FAISS index size:", index.ntotal)


FAISS index size: 2190


In [34]:
faiss.write_index(index, os.path.join(PROJECT_ROOT, "/faiss.index"))